In [0]:
# ============================================================
# GOLD LAYER - FACT_DELINQUENCY - CORRECTED VERSION
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# GET ENVIRONMENT FROM ADF
# ============================================================

try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from ADF: {env}")
except:
    env = "DEV"
    print(f"📌 Using default: {env}")

# ============================================================
# STORAGE ACCOUNT
# ============================================================

storage_account_name = "capstonestorage01"

# ============================================================
# CONTAINER NAMES BASED ON ENVIRONMENT
# ============================================================

if env == 'DEV':
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    silver_container = 'silver'
    gold_container = 'gold'

# ============================================================
# BUILD PATHS
# ============================================================

SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 SILVER Path: {SILVER}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

# ============================================================
# LOAD DATA
# ============================================================

print("📖 Loading data...")

df_deliq = spark.read.format("delta").load(f"{SILVER}/repayment_delinquency")
dim_account = spark.read.format("delta").load(f"{GOLD}/dim_account")
dim_customer = spark.read.format("delta").load(f"{GOLD}/dim_customer")

print(f"✅ Delinquency data: {df_deliq.count():,} rows")
print(f"✅ Dim Account: {dim_account.count():,} rows")
print(f"✅ Dim Customer: {dim_customer.count():,} rows")

# ============================================================
# JOIN & TRANSFORM - CREATE FACT_DELINQUENCY
# ============================================================

print("\n🔄 Joining and transforming...")

fact_delinquency = df_deliq \
    .join(dim_account.select(
        "account_id", "customer_id", "product_type",
        "credit_limit", "credit_tier"),
        "account_id", "left") \
    .join(dim_customer.select(
        "customer_id", "risk_segment", "city", "income_band"),
        "customer_id", "left") \
    .select(
        F.col("account_id"),
        F.col("customer_id"),
        F.col("due_date"),
        F.col("payment_date"),
        F.col("days_past_due"),
        F.col("overdue_amount"),
        F.col("delinquency_flag"),
        F.col("product_type"),
        F.col("credit_limit"),
        F.col("credit_tier"),
        F.col("risk_segment"),
        F.col("city"),
        F.col("income_band"),
        
        # DPD Bucket
        F.when(F.col("days_past_due") == 0, F.lit("Current"))
         .when(F.col("days_past_due") <= 30, F.lit("DPD 1-30"))
         .when(F.col("days_past_due") <= 60, F.lit("DPD 31-60"))
         .when(F.col("days_past_due") <= 90, F.lit("DPD 61-90"))
         .otherwise(F.lit("DPD 90+"))
         .alias("dpd_bucket"),
        
        # Overdue percentage of limit
        F.round(F.col("overdue_amount") / F.col("credit_limit") * 100, 2)
         .alias("overdue_pct_of_limit"),
        
        # Actual delay days
        F.datediff(F.col("payment_date"), F.col("due_date"))
         .alias("actual_delay_days"),
        
        F.current_timestamp().alias("gold_created_at")
    )

print(f"🔄 Fact delinquency rows: {fact_delinquency.count():,}")

# ============================================================
# WRITE TO GOLD
# ============================================================

print("\n💾 Writing to Gold...")

fact_delinquency.write \
    .format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .partitionBy("product_type") \
    .save(f"{GOLD}/fact_delinquency")

# ============================================================
# SUMMARY
# ============================================================

print(f"\n{'='*55}")
print(f"✅ fact_delinquency : {fact_delinquency.count():,} rows")
print(f"📁 Written to: {GOLD}/fact_delinquency")
print(f"🏭 Environment: {env}")
print(f"{'='*55}")